# Retrieval-Augmented Generation (RAG) in LangChain

RAG is an architecture pattern that augments LLM generation with external knowledge retrieved from a vector database at inference time.

**Why RAG exists:**
- LLMs have a knowledge cutoff date and cannot access private/proprietary data.
- Fine-tuning is expensive, slow, and impractical for frequently changing data.
- RAG reduces hallucinations by grounding responses in retrieved factual context.
- RAG enables domain-specific Q&A without retraining the model.

## RAG Architecture

### Ingestion Pipeline (Offline / Indexing Phase)

Raw Documents → Document Loader → Text Splitter → Embedding Model → Vector Store

### Query Pipeline (Online / Inference Phase)

User Query → Embedding Model → Vector Store Retrieval → Retrieved Context + Query → LLM → Generated Answer

### Key Insight

The ingestion pipeline runs once (or on schedule). The query pipeline runs per user request. Separating these two phases is critical for production RAG systems.

## Key Components & Their Roles

| Component | LangChain Class | Role in RAG |
|---|---|---|
| **Document Loader** | TextLoader, PyPDFLoader | Ingests raw files into Document objects with page_content and metadata. |
| **Text Splitter** | RecursiveCharacterTextSplitter | Splits large documents into semantically coherent chunks for embedding. |
| **Embedding Model** | GoogleGenerativeAIEmbeddings | Converts text chunks into dense numerical vectors for similarity search. |
| **Vector Store** | Chroma, FAISS | Stores and indexes embedding vectors for fast approximate nearest neighbor search. |
| **Retriever** | VectorStoreRetriever | Wraps vector store as an LCEL Runnable for prompt-in, documents-out search. |
| **Stuff Documents Chain** | create_stuff_documents_chain | Stuffs all retrieved document chunks into the LLM prompt context window. |
| **Retrieval Chain** | create_retrieval_chain | Composes retriever + stuff chain into an end-to-end RAG pipeline. |
| **History-Aware Retriever** | create_history_aware_retriever | Reformulates queries using chat history for multi-turn conversations. |

## Common Mistakes in RAG Applications

| Mistake | Why It Fails | Correct Approach |
|---|---|---|
| Using chunk_size too large (>2000) | Embedding quality degrades; irrelevant noise in context. | Use 500-1000 chars with 50-200 overlap. |
| Using chunk_overlap = 0 | Splits mid-sentence, losing cross-boundary context. | Set overlap to 10-20% of chunk_size. |
| Not preserving metadata during splitting | Lose traceability to source document. | Pass metadata through splitter pipeline. |
| Using predict() instead of invoke() | predict() is deprecated in modern LangChain. | Always use chain.invoke({"input": query}). |
| Retrieving too many chunks (k > 10) | Floods LLM context window; increases cost and latency. | Start with k=3-5 and tune based on quality. |
| Ignoring embedding model dimensions | Mismatched vector dimensions cause index errors. | Use consistent embedding model throughout pipeline. |
| Not handling empty retrieval results | LLM hallucinates when no relevant context is found. | Add fallback: "I don't have enough information." |
| Hardcoding file paths in loaders | Breaks portability across environments. | Use relative paths or environment variables. |

## Why This Architecture Was Chosen

1. **LCEL Composability**: create_retrieval_chain and create_stuff_documents_chain use LangChain Expression Language, making chains streamable, batchable, and inspectable.
2. **Separation of Concerns**: Document ingestion is decoupled from query-time retrieval, allowing independent scaling.
3. **Stuff Strategy**: For small-to-medium context windows, stuffing all retrieved chunks into a single prompt is the simplest and most predictable strategy.
4. **Chroma In-Memory**: Eliminates SQLite file lock issues during development while providing full vector search capabilities.

## Alternative Approaches

| Approach | When to Use | Trade-Off |
|---|---|---|
| **Map-Reduce Chain** | Very large document sets exceeding context window. | Higher latency; multiple LLM calls per query. |
| **Refine Chain** | Sequential document refinement needed. | Slower; each chunk refines previous answer. |
| **FAISS instead of Chroma** | High-performance local indexing with millions of vectors. | No built-in metadata filtering; requires manual persistence. |
| **Hybrid Search (BM25 + Dense)** | Technical documents with exact keyword matching needs. | More complex pipeline; requires rank fusion. |
| **Re-Ranking with Cross-Encoders** | Production systems needing precision over recall. | Additional model inference step; higher latency. |
| **Agentic RAG** | Dynamic tool selection and multi-step reasoning. | Significantly more complex; harder to debug. |

## Performance Optimizations

1. **Chunk Size Tuning**: Experiment with chunk_size (300-1000) and chunk_overlap (50-200) based on document structure. Smaller chunks improve retrieval precision; larger chunks provide richer context.
2. **Embedding Caching**: Cache embeddings to avoid recomputing on every restart. Use Chroma persist_directory or FAISS save_local/load_local.
3. **Retriever k Tuning**: Start with k=3, measure answer quality, and increase only if recall is too low.
4. **MMR Search**: Use search_type="mmr" to diversify retrieved chunks and reduce redundancy.
5. **Streaming Responses**: Use chain.stream() instead of chain.invoke() for real-time token-by-token output.
6. **Async Execution**: Use chain.ainvoke() for concurrent request handling in web applications.
7. **Metadata Filtering**: Pre-filter documents by source, date, or category before vector search to reduce noise.
8. **Contextual Compression**: Apply EmbeddingsFilter post-retrieval to remove low-relevance chunks before LLM generation.

## Interview Questions & Answers

**Q1: What is RAG and why is it preferred over fine-tuning?**
RAG retrieves external knowledge at inference time. Fine-tuning bakes knowledge into model weights. RAG is cheaper, faster to update, and doesn't require GPU retraining. Fine-tuning is better for style/behavior changes, not knowledge updates.

**Q2: What is the difference between create_stuff_documents_chain and create_retrieval_chain?**
create_stuff_documents_chain takes a list of documents and stuffs them into the LLM prompt. create_retrieval_chain composes a retriever with a stuff chain, automatically fetching documents before generation.

**Q3: Why use RecursiveCharacterTextSplitter over CharacterTextSplitter?**
RecursiveCharacterTextSplitter tries multiple separators (\n\n, \n, " ", "") in order, preserving semantic boundaries like paragraphs and sentences. CharacterTextSplitter uses only one separator.

**Q4: What happens if the retriever returns no relevant documents?**
The LLM receives an empty context and may hallucinate. Production systems should detect empty retrieval and return a fallback response like "I don't have enough information to answer this question."

**Q5: How does create_history_aware_retriever work?**
It takes the latest user question and chat history, then uses an LLM to reformulate the question into a standalone query that doesn't depend on conversational context. This reformulated query is then passed to the base retriever.

**Q6: What is the difference between similarity search and MMR search in retrieval?**
Similarity search returns the k most similar documents. MMR (Maximal Marginal Relevance) balances similarity with diversity, penalizing documents that are too similar to already-selected results.

**Q7: Why use invoke() instead of run() or predict()?**
run() and predict() are deprecated legacy APIs. invoke() is the standard LCEL interface that supports streaming, batching, async execution, and structured input/output.

**Q8: How would you handle a PDF with 500 pages in a RAG pipeline?**
Use PyPDFLoader to load all pages, RecursiveCharacterTextSplitter to chunk them, and persist embeddings to disk (Chroma persist_directory or FAISS save_local). At query time, load the pre-built index instead of re-embedding.

**Q9: What is chunk_overlap and why is it important?**
chunk_overlap creates overlapping text between consecutive chunks. Without it, information spanning chunk boundaries is lost. Typical values are 10-20% of chunk_size.

**Q10: How would you evaluate RAG quality in production?**
Use metrics like Faithfulness (is the answer grounded in context?), Answer Relevancy (does it address the question?), and Context Precision (are retrieved chunks relevant?). Tools like RAGAS and LangSmith provide automated evaluation.

**Q11: What is the Stuff strategy and when does it fail?**
The Stuff strategy concatenates all retrieved documents into a single prompt. It fails when total retrieved text exceeds the LLM context window. For very large retrievals, use Map-Reduce or Refine strategies instead.

**Q12: How do you add source attribution to RAG responses?**
Include source metadata (filename, page number) in retrieved Document objects. The retrieval chain returns these in result["context"], which can be displayed alongside the generated answer.